# CLASSIFICATIE

In [ ]:
from sklearn.tree import plot_tree
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor,RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import plot_tree
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor,RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, train_test_split




## TRAIN TEST SET

In [ ]:
df_all = pd.read_csv('goedeoutput.csv')
df_all = df_all.drop_duplicates(subset='file')
df_all.head()

In [ ]:
colomn_names = df_all.columns
colomn_names

# colomnen verwijderen
feature_names=['time', 'x', 'y', 'z', 'abs', 'weight', 'weather',
       'x_std', 'z_std', 'x_mean',
       'x_rms', 'z_rms',
       'z_spectral_flatness']
Y = df_all['roadtype']
X = df_all[feature_names]
X = pd.get_dummies(X, columns=['weather'], drop_first=True, dtype=int)

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## model

In [ ]:
scaler = StandardScaler()
x_train_scale = scaler.fit_transform(x_train)
x_test_scale = scaler.transform(x_test)
classifier = RandomForestClassifier(n_estimators=100, random_state=12)
classifier.fit(x_train_scale, y_train)
y_pred = classifier.predict(x_test_scale)
ac = accuracy_score(y_test,y_pred)

print(f'de accuracy is {ac}')

In [ ]:
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

correct_counts = results[results['Actual'] == results['Predicted']] \
                    .groupby('Actual') \
                    .size()

print(correct_counts)

percentages = correct_counts / results['Actual'].value_counts() * 100
print(percentages)

In [ ]:
conf_matrix = confusion_matrix(y_test,y_pred)
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted road type")
plt.ylabel("True road type")
plt.title("Confusion Matrix")
plt.show()

## met gridsearch

LET OP! gridsearch duurt 85 minuten

In [67]:
pipeline = make_pipeline(StandardScaler(), RandomForestClassifier()) #max_iter=1000, tol=1e-3))

param_grid = {
    'randomforestclassifier__n_estimators': [300, 500, 800],
    'randomforestclassifier__max_depth': [10, 20, 40, None],
    'randomforestclassifier__min_samples_split': [2, 5, 10],
    'randomforestclassifier__min_samples_leaf': [1, 2, 4],
    'randomforestclassifier__max_features': ['sqrt', 'log2'],
    'randomforestclassifier__criterion': ['gini', 'entropy']
}




grid_cv = GridSearchCV(pipeline, 
                       param_grid=param_grid,
                       n_jobs=1, 
                       cv=7, 
                       verbose=2) 

grid_cv.fit(x_train, y_train)

[CV] END randomforestclassifier__criterion=gini, randomforestclassifier__max_depth=10, randomforestclassifier__max_features=sqrt, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__min_samples_split=2, randomforestclassifier__n_estimators=800; total time=   1.8s
[CV] END randomforestclassifier__criterion=gini, randomforestclassifier__max_depth=10, randomforestclassifier__max_features=sqrt, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__min_samples_split=5, randomforestclassifier__n_estimators=300; total time=   0.6s
[CV] END randomforestclassifier__criterion=gini, randomforestclassifier__max_depth=10, randomforestclassifier__max_features=sqrt, randomforestclassifier__min_samples_leaf=1, randomforestclassifier__min_samples_split=5, randomforestclassifier__n_estimators=300; total time=   0.6s
[CV] END randomforestclassifier__criterion=gini, randomforestclassifier__max_depth=10, randomforestclassifier__max_features=sqrt, randomforestclassifier__min_sa

KeyboardInterrupt: 

In [ ]:
param_grid = {
    'randomforestclassifier__n_estimators': [300, 500],
    'randomforestclassifier__max_depth': [20, None],
    'randomforestclassifier__min_samples_split': [2, 5],
    'randomforestclassifier__min_samples_leaf': [1, 2],
    'randomforestclassifier__max_features': ['sqrt']
}

In [ ]:
print(f'For the training set, using K-fold CV the best estimator is: {grid_cv.best_estimator_}')
print(f'Leading to an accuracy of: {grid_cv.best_score_}')

In [ ]:
y_pred = grid_cv.best_estimator_.predict(x_test)
ass = accuracy_score(y_test,y_pred)
ass

In [ ]:
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

correct_counts = results[results['Actual'] == results['Predicted']] \
                    .groupby('Actual') \
                    .size()

print(correct_counts)

percentages = correct_counts / results['Actual'].value_counts() * 100
print(percentages)

In [ ]:
class_names = ['asphalt', 'bricks', 'grass', 'gravel', 'sand', 'wood']
conf_matrix = confusion_matrix(y_test,y_pred)
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",xticklabels=class_names,
    yticklabels=class_names)

plt.xlabel("Predicted road type")
plt.ylabel("True road type")
plt.title("Confusion Matrix")
plt.show()

## DOCUMENTATIE

https://www.geeksforgeeks.org/random-forest-classifier-using-scikit-learn/